In [1]:
import os
from pathlib import Path
import warnings

import pandas as pd
import numpy as np
import librosa, librosa.display
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

warnings.filterwarnings("ignore")

# =========================
# Config
# =========================

ROOT_DIR = Path("D:\WABAD")   # folder containing site folders
OUT_DIR  = ROOT_DIR / "_roi_plots"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Expected names inside each site folder
AUDIO_DIR_NAMES = ["recordings", "recording", "audio", "wav", "wavs"]
GT_DIR_NAMES    = ["Raven Pro annotations"]

AUDIO_EXTS = [".wav", ".flac", ".mp3"]

ZOOM_END = 60.0      # seconds shown in plot. Set to None for full duration
SR = None            # None keeps original sampling rate
HOP = 512
N_FFT = 2048
FMAX_HZ = 10000

SHOW = False         # True to display plots interactively

# Resume/skip behaviour
SKIP_EXISTING = True
FORCE_REBUILD = False

# Avoid treating failed/empty output files as completed
MIN_OUTPUT_BYTES = 10_000

SUMMARY_CSV = OUT_DIR / "roi_plot_summary.csv"


# =========================
# Helpers
# =========================

def find_subdir(parent: Path, possible_names: list[str]):
    """Find first matching subfolder by name, case-insensitive."""
    if not parent.exists():
        return None

    children = [p for p in parent.iterdir() if p.is_dir()]
    lookup = {p.name.lower(): p for p in children}

    for name in possible_names:
        if name.lower() in lookup:
            return lookup[name.lower()]

    return None


def _pick_col(df: pd.DataFrame, candidates: list[str], fallback_idx: int | None = None):
    """Pick a column by case-insensitive name, otherwise fallback to index."""
    cols_lower = {str(c).lower().strip(): c for c in df.columns}

    for cand in candidates:
        key = cand.lower().strip()
        if key in cols_lower:
            return cols_lower[key]

    if fallback_idx is not None and 0 <= fallback_idx < len(df.columns):
        return df.columns[fallback_idx]

    return None


def read_raven_gt(gt_path: Path):
    """
    Reads Raven-style ground-truth file with flexible separators and column names.
    Returns list of (start, end, low_f, high_f, species).
    """
    try:
        df = pd.read_csv(gt_path, sep=r"\t|,", engine="python")
    except Exception as e:
        print(f"[WARN] Could not read GT: {gt_path.name} ({e})")
        return []

    df.columns = [str(c).strip() for c in df.columns]

    start_col = _pick_col(
        df,
        ["begin time (s)", "begin time", "start time (s)", "start time", "start", "begin"],
        fallback_idx=3
    )
    end_col = _pick_col(
        df,
        ["end time (s)", "end time", "stop time (s)", "stop time", "end", "stop"],
        fallback_idx=4
    )
    lowf_col = _pick_col(
        df,
        ["low freq (hz)", "low freq", "low frequency (hz)", "low frequency", "min freq (hz)"],
        fallback_idx=5
    )
    highf_col = _pick_col(
        df,
        ["high freq (hz)", "high freq", "high frequency (hz)", "high frequency", "max freq (hz)"],
        fallback_idx=6
    )
    species_col = _pick_col(
        df,
        ["species", "species scientific name", "scientific name", "annotation", "label", "class"],
        fallback_idx=8
    )

    rois = []

    for _, r in df.iterrows():
        try:
            s = float(r[start_col]) if start_col else np.nan
            e = float(r[end_col]) if end_col else np.nan

            lf = float(r[lowf_col]) if lowf_col and pd.notna(r[lowf_col]) else 0.0
            hf = float(r[highf_col]) if highf_col and pd.notna(r[highf_col]) else float(FMAX_HZ)

            sp = str(r[species_col]).strip() if species_col and pd.notna(r[species_col]) else "Unknown"

            if np.isfinite(s) and np.isfinite(e) and e > s:
                rois.append((s, e, lf, hf, sp))

        except Exception:
            continue

    return rois


def find_matching_gt(wav_path: Path, gt_dir: Path):
    """
    Find matching Raven annotation file by basename.

    Example:
    file_001.wav -> file_001.txt
    """
    possible_exts = [".txt", ".csv", ".tsv"]

    # Exact basename match
    for ext in possible_exts:
        candidate = gt_dir / f"{wav_path.stem}{ext}"
        if candidate.exists():
            return candidate

    # Fallback: annotation file starts with audio stem
    for ext in possible_exts:
        candidates = sorted(gt_dir.glob(f"{wav_path.stem}*{ext}"))
        if candidates:
            return candidates[0]

    # Fallback: annotation file contains audio stem
    for ext in possible_exts:
        candidates = sorted(gt_dir.glob(f"*{wav_path.stem}*{ext}"))
        if candidates:
            return candidates[0]

    return None


def output_is_valid(path: Path, min_bytes: int = MIN_OUTPUT_BYTES):
    """
    Checks whether an existing output PNG looks complete enough to skip.
    """
    return path.exists() and path.is_file() and path.stat().st_size >= min_bytes


def load_completed_outputs(summary_csv: Path):
    """
    Loads previously completed outputs from roi_plot_summary.csv.

    Returns a set of output PNG paths that were successfully generated.
    This is used as a second resume mechanism in addition to checking
    whether the PNG exists on disk.
    """
    if not summary_csv.exists():
        return set()

    try:
        df = pd.read_csv(summary_csv)
    except Exception:
        return set()

    if "status" not in df.columns or "out_png" not in df.columns:
        return set()

    completed = df[
        (df["status"] == "ok") &
        (df["out_png"].notna()) &
        (df["out_png"].astype(str).str.len() > 0)
    ]["out_png"].astype(str).tolist()

    return set(completed)


def append_summary_row(row: dict, summary_csv: Path):
    """
    Appends a single row to the summary CSV immediately.

    This makes the script more robust because progress is saved after
    each recording rather than only at the end.
    """
    row_df = pd.DataFrame([row])

    write_header = not summary_csv.exists()

    row_df.to_csv(
        summary_csv,
        mode="a",
        header=write_header,
        index=False
    )


def plot_spectrogram_with_rois(wav_path: Path, rois, out_png: Path, site_name: str):
    """Plots and saves spectrogram with ROI rectangles and species legend."""
    y, sr = librosa.load(str(wav_path), sr=SR, mono=True)

    S_full = librosa.stft(y, n_fft=N_FFT, hop_length=HOP)
    S_db = librosa.amplitude_to_db(np.abs(S_full), ref=np.max)

    species_list = sorted(set([sp for _, _, _, _, sp in rois]))

    colors = plt.cm.tab20(np.linspace(0, 1, max(1, len(species_list))))
    color_map = {sp: colors[i % len(colors)] for i, sp in enumerate(species_list)}

    plt.figure(figsize=(15, 6))

    librosa.display.specshow(
        S_db,
        sr=sr,
        hop_length=HOP,
        x_axis="time",
        y_axis="hz"
    )

    plt.ylim(0, FMAX_HZ)

    if ZOOM_END is not None:
        plt.xlim(0, ZOOM_END)

    ax = plt.gca()

    for (s, e, lf, hf, sp) in rois:
        if ZOOM_END is not None and s > ZOOM_END:
            continue

        e_vis = min(e, ZOOM_END) if ZOOM_END is not None else e

        if e_vis <= s:
            continue

        rect = mpatches.Rectangle(
            (s, lf),
            e_vis - s,
            max(0.0, hf - lf),
            facecolor=color_map.get(sp, (1, 0, 0, 0.3)),
            alpha=0.3,
            edgecolor=color_map.get(sp, (1, 0, 0, 1.0)),
            lw=1.0,
        )

        ax.add_patch(rect)

    if species_list:
        legend_patches = [
            mpatches.Patch(color=color_map[sp], label=sp)
            for sp in species_list
        ]

        plt.legend(
            handles=legend_patches,
            loc="center left",
            bbox_to_anchor=(1, 0.5),
            fontsize=8,
            frameon=True,
        )

    plt.title(f"{site_name} | Spectrogram + Ground Truth ROIs\n{wav_path.name}")
    plt.xlabel("Time (s)")
    plt.ylabel("Frequency (Hz)")

    plt.tight_layout()
    plt.savefig(out_png, dpi=200, bbox_inches="tight")

    if SHOW:
        plt.show()

    plt.close()


# =========================
# Main: folder-of-folders scan
# =========================

completed_outputs = load_completed_outputs(SUMMARY_CSV)

site_dirs = sorted([p for p in ROOT_DIR.iterdir() if p.is_dir() and not p.name.startswith("_")])

if not site_dirs:
    raise FileNotFoundError(f"No site folders found in: {ROOT_DIR.resolve()}")

print(f"Found {len(site_dirs)} site folders.")

total_ok = 0
total_skip = 0
total_err = 0
total_no_gt = 0

for site_dir in site_dirs:
    site_name = site_dir.name

    audio_dir = find_subdir(site_dir, AUDIO_DIR_NAMES)
    gt_dir = find_subdir(site_dir, GT_DIR_NAMES)

    if audio_dir is None:
        print(f"[SKIP] {site_name}: no audio/recordings folder found.")
        continue

    if gt_dir is None:
        print(f"[SKIP] {site_name}: no annotations/Raven folder found.")
        continue

    site_out_dir = OUT_DIR / site_name
    site_out_dir.mkdir(parents=True, exist_ok=True)

    wav_files = []
    for ext in AUDIO_EXTS:
        wav_files.extend(audio_dir.rglob(f"*{ext}"))

    wav_files = sorted(wav_files)

    print(f"\n[{site_name}] Found {len(wav_files)} audio files.")
    print(f"  Audio folder: {audio_dir}")
    print(f"  GT folder:    {gt_dir}")

    if not wav_files:
        row = {
            "site": site_name,
            "wav": "",
            "gt": "",
            "n_rois": 0,
            "n_species": 0,
            "status": "no_audio",
            "out_png": "",
        }
        append_summary_row(row, SUMMARY_CSV)
        continue

    for wav_path in wav_files:
        gt_path = find_matching_gt(wav_path, gt_dir)
        out_png = site_out_dir / f"{wav_path.stem}_rois.png"

        # ---------------------------------------------------------
        # Resume / skip check
        # ---------------------------------------------------------
        if SKIP_EXISTING and not FORCE_REBUILD:
            output_path_str = str(out_png)

            if output_is_valid(out_png) or output_path_str in completed_outputs:
                print(f"  [DONE] Skipping existing output: {out_png.name}")

                total_skip += 1

                # Optional: record the skip event in the summary
                row = {
                    "site": site_name,
                    "wav": wav_path.name,
                    "gt": Path(gt_path).name if gt_path else "",
                    "n_rois": "",
                    "n_species": "",
                    "status": "skipped_existing",
                    "out_png": str(out_png),
                }
                append_summary_row(row, SUMMARY_CSV)

                continue

        if gt_path is None:
            print(f"  [SKIP] No GT found for {wav_path.name}")

            total_no_gt += 1

            row = {
                "site": site_name,
                "wav": wav_path.name,
                "gt": "",
                "n_rois": 0,
                "n_species": 0,
                "status": "no_gt",
                "out_png": "",
            }
            append_summary_row(row, SUMMARY_CSV)

            continue

        rois = read_raven_gt(gt_path)

        try:
            plot_spectrogram_with_rois(
                wav_path=wav_path,
                rois=rois,
                out_png=out_png,
                site_name=site_name
            )

            status = "ok"
            total_ok += 1

            print(f"  [OK] {wav_path.name} + {gt_path.name} -> {out_png.name}")

        except Exception as e:
            status = f"error: {e}"
            total_err += 1

            print(f"  [ERR] {wav_path.name}: {e}")

        species_set = set([sp for _, _, _, _, sp in rois])

        row = {
            "site": site_name,
            "wav": wav_path.name,
            "gt": gt_path.name,
            "n_rois": len(rois),
            "n_species": len(species_set),
            "status": status,
            "out_png": str(out_png),
        }

        append_summary_row(row, SUMMARY_CSV)


print("\nFinished ROI plotting.")
print(f"  Generated:        {total_ok}")
print(f"  Skipped existing: {total_skip}")
print(f"  Missing GT:       {total_no_gt}")
print(f"  Errors:           {total_err}")
print(f"\nSummary written incrementally to: {SUMMARY_CSV.resolve()}")

<>:17: SyntaxWarning: "\W" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\W"? A raw string is also an option.
<>:17: SyntaxWarning: "\W" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\W"? A raw string is also an option.
C:\Users\Dr. Tom\AppData\Local\Temp\ipykernel_25288\2650381986.py:17: SyntaxWarning: "\W" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\W"? A raw string is also an option.
  ROOT_DIR = Path("D:\WABAD")   # folder containing site folders


Found 9 site folders.

[BAM (1)] Found 118 audio files.
  Audio folder: D:\WABAD\BAM (1)\Recordings
  GT folder:    D:\WABAD\BAM (1)\Raven Pro annotations
  [DONE] Skipping existing output: BAM_20151116_060801_rois.png
  [DONE] Skipping existing output: BAM_20151116_060810_rois.png
  [DONE] Skipping existing output: BAM_20151116_060811_rois.png
  [DONE] Skipping existing output: BAM_20151116_060812_rois.png
  [DONE] Skipping existing output: BAM_20151116_060813_rois.png
  [DONE] Skipping existing output: BAM_20151116_060815_rois.png
  [DONE] Skipping existing output: BAM_20151116_070812_rois.png
  [DONE] Skipping existing output: BAM_20151116_070813_rois.png
  [DONE] Skipping existing output: BAM_20151116_070814_rois.png
  [DONE] Skipping existing output: BAM_20151116_070815_rois.png
  [DONE] Skipping existing output: BAM_20151116_070817_rois.png
  [DONE] Skipping existing output: BAM_20151116_070819_rois.png
  [DONE] Skipping existing output: BAM_20151116_080812_rois.png
  [DONE] Skip

In [7]:
from __future__ import annotations

import json
import re
import shutil
import subprocess
from pathlib import Path
from urllib.parse import parse_qs, urlparse

import numpy as np
import pandas as pd

# =============================================================================
# Configuration
# =============================================================================

ROOT_DIR = Path(r"D:\WABAD\SERF")
CSV_PATH = ROOT_DIR / "serf_acoustic_study_397-20260715-021743.csv"
AUDIO_ROOT = ROOT_DIR

OUT_DIR = ROOT_DIR / "_serf_labelled_60s_audio"
MANIFEST_CSV = OUT_DIR / "serf_60s_segment_manifest.csv"

AUDIO_EXTS = {".wav", ".flac", ".mp3", ".ogg", ".m4a"}
SEGMENT_SECONDS = 60.0
INCLUDE_PARTIAL_FINAL_SEGMENT = False

# -----------------------------------------------------------------------------
# GT filtering
# -----------------------------------------------------------------------------
# "any": keep a segment whenever at least one valid GT event overlaps it.
#
# "significant": keep only when ALL configured significance thresholds below
# are satisfied. This is useful when isolated or extremely short annotations
# should not be retained.
KEEP_RULE = "any"  # "any" or "significant"

# Applied only when KEEP_RULE == "significant".
MIN_EVENT_COUNT = 1
MIN_DISTINCT_LABELS = 1
MIN_LABELLED_SECONDS = 3.0
MIN_LABELLED_FRACTION = 0.05   # 0.05 = 3 seconds of a 60-second segment

# Unlabelled GT rows are excluded from the decision by default.
EXCLUDE_UNLABELLED_GT = True
UNLABELLED_NAME = "Unlabelled"
LABEL_COLUMNS = ["common_name_tags", "species_name_tags", "other_tags"]

# Audio files that cannot be matched to the GT CSV cannot be filtered reliably,
# so they are skipped rather than exported.
SKIP_UNMATCHED_AUDIO = True

# -----------------------------------------------------------------------------
# Output
# -----------------------------------------------------------------------------
# FLAC is lossless and smaller than WAV. Alternatives: "wav" or "mp3".
OUTPUT_FORMAT = "flac"
WRITE_CLIP_ANNOTATION_CSV = True

SKIP_EXISTING = True
FORCE_REBUILD = False
MIN_OUTPUT_BYTES = 1_000

# Keep source sample rate and channel count. The codec settings below only
# determine the output container/encoding.
FFMPEG_EXE = None
FFPROBE_EXE = None


# =============================================================================
# CSV loading
# =============================================================================

REQUIRED_COLUMNS = {
    "audio_event_id",
    "audio_recording_id",
    "audio_recording_uuid",
    "audio_recording_start_datetime_australia_brisbane_10_00",
    "site_id",
    "site_name",
    "event_start_seconds",
    "event_end_seconds",
    "low_frequency_hertz",
    "high_frequency_hertz",
    "listen_url",
}


def clean_tag_component(text: str) -> str:
    text = str(text).strip()
    text = re.sub(r"^\d+:", "", text)
    text = re.sub(r":general$", "", text, flags=re.IGNORECASE)
    text = text.replace("_", " ")
    return re.sub(r"\s+", " ", text).strip()


def clean_tag_value(value) -> str | None:
    if pd.isna(value):
        return None

    labels: list[str] = []
    for part in str(value).split("|"):
        label = clean_tag_component(part)
        if label and label.lower() not in {"nan", "none"} and label not in labels:
            labels.append(label)

    return " + ".join(labels) if labels else None


def choose_event_label(row: pd.Series) -> str:
    for column in LABEL_COLUMNS:
        label = clean_tag_value(row.get(column))
        if label:
            return label
    return UNLABELLED_NAME


def parse_listen_window(url: str) -> tuple[float, float]:
    try:
        query = parse_qs(urlparse(str(url)).query)
        start = float(query["start"][0])
        end = float(query["end"][0])
        if end > start:
            return start, end
    except (KeyError, TypeError, ValueError):
        pass
    return np.nan, np.nan


def resolve_csv_path(configured_path: Path) -> Path:
    if configured_path.exists():
        return configured_path

    candidates = sorted(
        configured_path.parent.glob("serf_acoustic_study_*.csv"),
        key=lambda path: path.stat().st_mtime,
        reverse=True,
    )
    if not candidates:
        raise FileNotFoundError(
            f"SERF annotation CSV not found: {configured_path}. "
            "No serf_acoustic_study_*.csv fallback was found."
        )

    selected = candidates[0]
    print(f"Configured CSV was not found; using: {selected}")
    return selected


def load_serf_annotations(csv_path: Path) -> pd.DataFrame:
    if not csv_path.exists():
        raise FileNotFoundError(f"SERF CSV not found: {csv_path}")

    header = pd.read_csv(csv_path, nrows=0)
    missing = REQUIRED_COLUMNS.difference(header.columns)
    if missing:
        raise ValueError(
            "The CSV is missing required columns: " + ", ".join(sorted(missing))
        )

    available_label_columns = [
        column for column in LABEL_COLUMNS if column in header.columns
    ]
    usecols = sorted(REQUIRED_COLUMNS.union(available_label_columns))
    df = pd.read_csv(csv_path, usecols=usecols, low_memory=False)

    for column in LABEL_COLUMNS:
        if column not in df.columns:
            df[column] = np.nan

    numeric_columns = [
        "audio_event_id",
        "audio_recording_id",
        "site_id",
        "event_start_seconds",
        "event_end_seconds",
        "low_frequency_hertz",
        "high_frequency_hertz",
    ]
    for column in numeric_columns:
        df[column] = pd.to_numeric(df[column], errors="coerce")

    df = df.dropna(
        subset=[
            "audio_event_id",
            "audio_recording_id",
            "event_start_seconds",
            "event_end_seconds",
        ]
    ).copy()
    df = df[df["event_end_seconds"] > df["event_start_seconds"]].copy()

    df["audio_event_id"] = df["audio_event_id"].astype(np.int64)
    df["audio_recording_id"] = df["audio_recording_id"].astype(np.int64)
    df["site_id"] = df["site_id"].astype("Int64")

    windows = df["listen_url"].map(parse_listen_window)
    df["window_start"] = [item[0] for item in windows]
    df["window_end"] = [item[1] for item in windows]

    df["label"] = df.apply(choose_event_label, axis=1)
    df["audio_recording_uuid"] = (
        df["audio_recording_uuid"].fillna("").astype(str).str.lower().str.strip()
    )
    df["site_name"] = df["site_name"].fillna("Unknown site").astype(str).str.strip()

    return df.sort_values(
        ["audio_recording_id", "event_start_seconds", "event_end_seconds"]
    ).reset_index(drop=True)


# =============================================================================
# Audio matching
# =============================================================================


def normalise_filename(text: str) -> str:
    return re.sub(r"[^a-z0-9]+", "_", text.lower()).strip("_")


def datetime_tokens(value: str) -> set[str]:
    if not value or str(value).lower() == "nan":
        return set()

    raw = str(value).strip()
    tokens = {
        normalise_filename(raw),
        normalise_filename(raw.replace(":", "")),
        normalise_filename(raw.replace("-", "").replace(":", "")),
    }

    try:
        dt = pd.to_datetime(raw)
        tokens.update(
            {
                normalise_filename(dt.strftime("%Y%m%dT%H%M%S%z")),
                normalise_filename(dt.strftime("%Y%m%d_%H%M%S")),
                normalise_filename(dt.strftime("%Y-%m-%d_%H-%M-%S")),
            }
        )
    except (TypeError, ValueError):
        pass

    return {token for token in tokens if token}


def build_lookup_tables(df: pd.DataFrame) -> dict:
    recording_groups = {
        int(recording_id): group.reset_index(drop=True)
        for recording_id, group in df.groupby("audio_recording_id", sort=False)
    }

    uuid_to_recording: dict[str, int] = {}
    datetime_to_recordings: dict[str, set[int]] = {}

    recording_meta = (
        df.groupby("audio_recording_id", as_index=False)
        .first()[
            [
                "audio_recording_id",
                "audio_recording_uuid",
                "audio_recording_start_datetime_australia_brisbane_10_00",
            ]
        ]
    )

    for row in recording_meta.itertuples(index=False):
        recording_id = int(row.audio_recording_id)
        uuid = str(row.audio_recording_uuid).lower().strip()
        if uuid:
            uuid_to_recording[uuid] = recording_id

        dt_value = getattr(
            row,
            "audio_recording_start_datetime_australia_brisbane_10_00",
        )
        for token in datetime_tokens(dt_value):
            datetime_to_recordings.setdefault(token, set()).add(recording_id)

    event_to_recording = dict(
        zip(df["audio_event_id"].astype(int), df["audio_recording_id"].astype(int))
    )
    event_to_window = dict(
        zip(
            df["audio_event_id"].astype(int),
            zip(df["window_start"], df["window_end"]),
        )
    )

    return {
        "recording_groups": recording_groups,
        "recording_ids": set(recording_groups),
        "uuid_to_recording": uuid_to_recording,
        "datetime_to_recordings": datetime_to_recordings,
        "event_to_recording": event_to_recording,
        "event_to_window": event_to_window,
    }


def integer_tokens(stem: str) -> list[int]:
    return [int(value) for value in re.findall(r"(?<!\d)(\d+)(?!\d)", stem)]


def match_audio_to_recording(
    audio_path: Path,
    lookups: dict,
) -> tuple[int | None, int | None, str]:
    stem_lower = audio_path.stem.lower()
    stem_normal = normalise_filename(audio_path.stem)

    for uuid, recording_id in lookups["uuid_to_recording"].items():
        if uuid in stem_lower:
            return recording_id, None, "recording_uuid"

    explicit_event = re.search(r"(?:event|audio_event)[_-]?(\d+)", stem_lower)
    if explicit_event:
        event_id = int(explicit_event.group(1))
        recording_id = lookups["event_to_recording"].get(event_id)
        if recording_id is not None:
            return recording_id, event_id, "explicit_event_id"

    explicit_recording = re.search(
        r"(?:recording|audio_recording|rec)[_-]?(\d+)", stem_lower
    )
    if explicit_recording:
        recording_id = int(explicit_recording.group(1))
        if recording_id in lookups["recording_ids"]:
            return recording_id, None, "explicit_recording_id"

    numbers = integer_tokens(stem_lower)
    recording_hits = [n for n in numbers if n in lookups["recording_ids"]]
    if len(set(recording_hits)) == 1:
        return recording_hits[0], None, "recording_id"

    event_hits = [n for n in numbers if n in lookups["event_to_recording"]]
    event_recordings = {lookups["event_to_recording"][n] for n in event_hits}
    if len(event_recordings) == 1:
        event_id = event_hits[0]
        return next(iter(event_recordings)), event_id, "event_id"

    datetime_hits: set[int] = set()
    for token, recording_ids in lookups["datetime_to_recordings"].items():
        if token and token in stem_normal:
            datetime_hits.update(recording_ids)

    if len(datetime_hits) == 1:
        return next(iter(datetime_hits)), None, "recording_datetime"

    return None, None, "unmatched"


# =============================================================================
# Audio timing
# =============================================================================


def known_windows(recording_df: pd.DataFrame) -> list[tuple[float, float]]:
    windows = (
        recording_df[["window_start", "window_end"]]
        .dropna()
        .drop_duplicates()
        .sort_values(["window_start", "window_end"])
    )
    return [tuple(map(float, row)) for row in windows.to_numpy()]


def find_window_numbers_in_filename(
    audio_path: Path,
    recording_df: pd.DataFrame,
) -> tuple[float, float] | None:
    stem = audio_path.stem.lower()

    patterns = [
        r"start[_-]?(\d+(?:\.\d+)?).*?end[_-]?(\d+(?:\.\d+)?)",
        r"from[_-]?(\d+(?:\.\d+)?).*?to[_-]?(\d+(?:\.\d+)?)",
    ]
    for pattern in patterns:
        match = re.search(pattern, stem)
        if match:
            start, end = map(float, match.groups())
            if end > start:
                return start, end

    match = re.search(
        r"offset[_-]?(\d+(?:\.\d+)?).*?(?:duration|dur)[_-]?(\d+(?:\.\d+)?)",
        stem,
    )
    if match:
        start, duration = map(float, match.groups())
        if duration > 0:
            return start, start + duration

    numeric_values = [
        float(value)
        for value in re.findall(r"(?<!\d)(\d+(?:\.\d+)?)(?!\d)", stem)
    ]
    rounded_values = {round(value, 4) for value in numeric_values}

    matches: list[tuple[float, float]] = []
    for start, end in known_windows(recording_df):
        if round(start, 4) in rounded_values and round(end, 4) in rounded_values:
            matches.append((start, end))

    return matches[0] if len(matches) == 1 else None


def infer_absolute_origin(
    audio_path: Path,
    recording_df: pd.DataFrame,
    matched_event_id: int | None,
    event_to_window: dict,
    audio_duration: float,
) -> tuple[float, str]:
    # Two-hour/full recordings begin at recording second zero.
    is_short_clip = audio_duration <= SEGMENT_SECONDS + 3.0
    if not is_short_clip:
        return 0.0, "recording_start"

    if matched_event_id is not None:
        candidate = event_to_window.get(matched_event_id)
        if candidate and np.isfinite(candidate[0]) and np.isfinite(candidate[1]):
            return float(candidate[0]), "event_listen_window"

    candidate = find_window_numbers_in_filename(audio_path, recording_df)
    if candidate is not None:
        return float(candidate[0]), "filename_window"

    windows = known_windows(recording_df)
    if len(windows) == 1:
        return float(windows[0][0]), "single_known_window"

    return 0.0, "unknown_short_clip_origin"


def iter_segments(
    audio_duration: float,
) -> list[tuple[int, float, float]]:
    if not np.isfinite(audio_duration) or audio_duration <= 0:
        return []
    if SEGMENT_SECONDS <= 0:
        raise ValueError("SEGMENT_SECONDS must be greater than zero.")

    full_count = int(np.floor(audio_duration / SEGMENT_SECONDS))
    segments = [
        (index, index * SEGMENT_SECONDS, (index + 1) * SEGMENT_SECONDS)
        for index in range(full_count)
    ]

    remainder = audio_duration - full_count * SEGMENT_SECONDS
    if INCLUDE_PARTIAL_FINAL_SEGMENT and remainder > 1e-6:
        segments.append((full_count, full_count * SEGMENT_SECONDS, audio_duration))

    if not segments and INCLUDE_PARTIAL_FINAL_SEGMENT and audio_duration > 0:
        segments.append((0, 0.0, audio_duration))

    return segments


# =============================================================================
# GT qualification
# =============================================================================


def overlapping_gt(
    recording_df: pd.DataFrame,
    absolute_start: float,
    absolute_end: float,
) -> pd.DataFrame:
    events = recording_df[
        (recording_df["event_end_seconds"] > absolute_start)
        & (recording_df["event_start_seconds"] < absolute_end)
    ].copy()

    if EXCLUDE_UNLABELLED_GT:
        events = events[events["label"] != UNLABELLED_NAME].copy()

    if events.empty:
        return events

    events["clip_start_seconds"] = (
        events["event_start_seconds"].clip(lower=absolute_start, upper=absolute_end)
        - absolute_start
    )
    events["clip_end_seconds"] = (
        events["event_end_seconds"].clip(lower=absolute_start, upper=absolute_end)
        - absolute_start
    )
    events["overlap_seconds"] = (
        events["clip_end_seconds"] - events["clip_start_seconds"]
    ).clip(lower=0.0)
    return events[events["overlap_seconds"] > 0].copy()


def interval_union_seconds(
    starts: pd.Series,
    ends: pd.Series,
) -> float:
    intervals = sorted(
        (float(start), float(end))
        for start, end in zip(starts, ends)
        if np.isfinite(start) and np.isfinite(end) and end > start
    )
    if not intervals:
        return 0.0

    total = 0.0
    current_start, current_end = intervals[0]
    for start, end in intervals[1:]:
        if start <= current_end:
            current_end = max(current_end, end)
        else:
            total += current_end - current_start
            current_start, current_end = start, end
    total += current_end - current_start
    return total


def evaluate_segment(
    events: pd.DataFrame,
    segment_duration: float,
) -> dict:
    if events.empty:
        metrics = {
            "event_count": 0,
            "distinct_label_count": 0,
            "labels": "",
            "labelled_seconds": 0.0,
            "labelled_fraction": 0.0,
        }
    else:
        labelled_seconds = interval_union_seconds(
            events["clip_start_seconds"],
            events["clip_end_seconds"],
        )
        labels = sorted(events["label"].astype(str).unique())
        metrics = {
            "event_count": int(len(events)),
            "distinct_label_count": int(len(labels)),
            "labels": " | ".join(labels),
            "labelled_seconds": float(labelled_seconds),
            "labelled_fraction": (
                float(labelled_seconds / segment_duration)
                if segment_duration > 0
                else 0.0
            ),
        }

    rule = KEEP_RULE.lower().strip()
    if rule == "any":
        keep = metrics["event_count"] >= 1
        reason = "has_gt" if keep else "no_gt"
    elif rule == "significant":
        checks = {
            "event_count": metrics["event_count"] >= MIN_EVENT_COUNT,
            "distinct_labels": (
                metrics["distinct_label_count"] >= MIN_DISTINCT_LABELS
            ),
            "labelled_seconds": (
                metrics["labelled_seconds"] >= MIN_LABELLED_SECONDS
            ),
            "labelled_fraction": (
                metrics["labelled_fraction"] >= MIN_LABELLED_FRACTION
            ),
        }
        keep = all(checks.values())
        failed = [name for name, passed in checks.items() if not passed]
        reason = "meets_significance_thresholds" if keep else "below_" + "+".join(failed)
    else:
        raise ValueError("KEEP_RULE must be either 'any' or 'significant'.")

    return {**metrics, "keep": keep, "decision_reason": reason}


# =============================================================================
# ffmpeg/ffprobe
# =============================================================================


def resolve_executable(configured, executable_name: str) -> str | None:
    if configured:
        configured_path = Path(configured)
        return str(configured_path) if configured_path.exists() else None
    return shutil.which(executable_name)


def probe_audio_duration(audio_path: Path) -> float:
    ffprobe = resolve_executable(FFPROBE_EXE, "ffprobe")
    if ffprobe is None:
        raise RuntimeError(
            "ffprobe was not found. Install ffmpeg and ensure ffprobe is on PATH, "
            "or set FFPROBE_EXE explicitly."
        )

    command = [
        ffprobe,
        "-v", "error",
        "-show_entries", "format=duration",
        "-of", "json",
        str(audio_path),
    ]
    result = subprocess.run(command, capture_output=True, text=True, check=False)
    if result.returncode != 0:
        raise RuntimeError(
            f"ffprobe failed ({result.returncode}): "
            f"{result.stderr.strip() or 'no error text'}"
        )

    payload = json.loads(result.stdout)
    duration = float(payload.get("format", {}).get("duration", "nan"))
    if not np.isfinite(duration) or duration <= 0:
        raise RuntimeError("ffprobe did not return a valid audio duration.")
    return duration


def output_extension() -> str:
    fmt = OUTPUT_FORMAT.lower().strip().lstrip(".")
    if fmt not in {"flac", "wav", "mp3"}:
        raise ValueError("OUTPUT_FORMAT must be 'flac', 'wav', or 'mp3'.")
    return "." + fmt


def extract_audio_segment(
    audio_path: Path,
    local_start: float,
    duration: float,
    out_audio: Path,
) -> None:
    ffmpeg = resolve_executable(FFMPEG_EXE, "ffmpeg")
    if ffmpeg is None:
        raise RuntimeError(
            "ffmpeg was not found. Install ffmpeg and ensure it is on PATH, "
            "or set FFMPEG_EXE explicitly."
        )

    fmt = OUTPUT_FORMAT.lower().strip().lstrip(".")
    codec_args: list[str]
    if fmt == "flac":
        codec_args = ["-c:a", "flac", "-compression_level", "5"]
    elif fmt == "wav":
        codec_args = ["-c:a", "pcm_s16le"]
    elif fmt == "mp3":
        codec_args = ["-c:a", "libmp3lame", "-q:a", "2"]
    else:
        raise ValueError("OUTPUT_FORMAT must be 'flac', 'wav', or 'mp3'.")

    out_audio.parent.mkdir(parents=True, exist_ok=True)
    temp_audio = out_audio.with_name(out_audio.stem + ".partial" + out_audio.suffix)
    if temp_audio.exists():
        temp_audio.unlink()

    command = [
        ffmpeg,
        "-v", "error",
        "-ss", f"{max(0.0, local_start):.6f}",
        "-t", f"{max(0.0, duration):.6f}",
        "-i", str(audio_path),
        "-map", "0:a:0",
        "-vn",
        *codec_args,
        "-y",
        str(temp_audio),
    ]
    result = subprocess.run(command, capture_output=True, text=True, check=False)
    if result.returncode != 0:
        temp_audio.unlink(missing_ok=True)
        raise RuntimeError(
            f"ffmpeg extraction failed ({result.returncode}): "
            f"{result.stderr.strip() or 'no error text'}"
        )

    if not temp_audio.exists() or temp_audio.stat().st_size < MIN_OUTPUT_BYTES:
        temp_audio.unlink(missing_ok=True)
        raise RuntimeError("ffmpeg produced a missing or unexpectedly small segment.")

    temp_audio.replace(out_audio)


# =============================================================================
# Output helpers
# =============================================================================


def safe_path_component(value: str) -> str:
    value = re.sub(r"[<>:\"/\\|?*]+", "_", str(value)).strip()
    return value or "Unknown"


def output_is_valid(path: Path) -> bool:
    return path.exists() and path.is_file() and path.stat().st_size >= MIN_OUTPUT_BYTES


def append_manifest_row(row: dict) -> None:
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    pd.DataFrame([row]).to_csv(
        MANIFEST_CSV,
        mode="a",
        header=not MANIFEST_CSV.exists(),
        index=False,
    )


def write_clip_annotations(events: pd.DataFrame, out_csv: Path) -> None:
    columns = [
        "audio_event_id",
        "label",
        "clip_start_seconds",
        "clip_end_seconds",
        "overlap_seconds",
        "low_frequency_hertz",
        "high_frequency_hertz",
        "event_start_seconds",
        "event_end_seconds",
    ]
    available = [column for column in columns if column in events.columns]
    out_csv.parent.mkdir(parents=True, exist_ok=True)
    events[available].to_csv(out_csv, index=False)


def discover_audio_files(audio_root: Path) -> list[Path]:
    if not audio_root.exists():
        raise FileNotFoundError(f"Audio root not found: {audio_root}")

    out_resolved = OUT_DIR.resolve()
    files: list[Path] = []
    for path in audio_root.rglob("*"):
        if not path.is_file() or path.suffix.lower() not in AUDIO_EXTS:
            continue
        try:
            if out_resolved in path.resolve().parents:
                continue
        except OSError:
            pass
        files.append(path)
    return sorted(files)


# =============================================================================
# Main
# =============================================================================


def main() -> None:
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    output_extension()  # validate configuration before processing

    csv_path = resolve_csv_path(CSV_PATH)
    print(f"Loading SERF annotations: {csv_path}")
    annotations = load_serf_annotations(csv_path)
    lookups = build_lookup_tables(annotations)

    print(
        f"Loaded {len(annotations):,} events from "
        f"{annotations['audio_recording_id'].nunique():,} recordings and "
        f"{annotations['site_name'].nunique():,} sites."
    )
    print(
        f"Segment length: {SEGMENT_SECONDS:g} s | keep rule: {KEEP_RULE} | "
        f"output: {OUTPUT_FORMAT.upper()}"
    )

    audio_files = discover_audio_files(AUDIO_ROOT)
    if not audio_files:
        raise FileNotFoundError(f"No supported audio files found below: {AUDIO_ROOT}")
    print(f"Found {len(audio_files):,} audio files.")

    totals = {
        "kept": 0,
        "discarded": 0,
        "skipped_existing": 0,
        "unmatched_files": 0,
        "errors": 0,
    }

    for file_index, audio_path in enumerate(audio_files, start=1):
        recording_id, matched_event_id, match_method = match_audio_to_recording(
            audio_path,
            lookups,
        )

        if recording_id is None:
            totals["unmatched_files"] += 1
            print(
                f"\n[{file_index}/{len(audio_files)}] [UNMATCHED] "
                f"{audio_path.name}: skipped because no GT recording could be identified."
            )
            append_manifest_row(
                {
                    "audio_file": str(audio_path),
                    "recording_id": "",
                    "site": "",
                    "match_method": match_method,
                    "segment_index": "",
                    "local_start": "",
                    "local_end": "",
                    "absolute_start": "",
                    "absolute_end": "",
                    "event_count": 0,
                    "distinct_label_count": 0,
                    "labels": "",
                    "labelled_seconds": 0.0,
                    "labelled_fraction": 0.0,
                    "decision": "unmatched_audio_skipped",
                    "decision_reason": "recording_not_in_gt_csv",
                    "status": "unmatched_audio",
                    "out_audio": "",
                    "annotation_csv": "",
                }
            )
            if SKIP_UNMATCHED_AUDIO:
                continue

        recording_df = lookups["recording_groups"].get(recording_id)
        if recording_df is None:
            continue

        site_name = str(recording_df["site_name"].iloc[0])

        try:
            audio_duration = probe_audio_duration(audio_path)
        except Exception as exc:
            totals["errors"] += 1
            print(f"[ERROR] Could not probe {audio_path.name}: {type(exc).__name__}: {exc}")
            append_manifest_row(
                {
                    "audio_file": str(audio_path),
                    "recording_id": recording_id,
                    "site": site_name,
                    "match_method": match_method,
                    "segment_index": "",
                    "decision": "error",
                    "decision_reason": "probe_failed",
                    "status": f"probe_error: {type(exc).__name__}: {exc}",
                    "out_audio": "",
                    "annotation_csv": "",
                }
            )
            continue

        absolute_origin, origin_method = infer_absolute_origin(
            audio_path=audio_path,
            recording_df=recording_df,
            matched_event_id=matched_event_id,
            event_to_window=lookups["event_to_window"],
            audio_duration=audio_duration,
        )
        segments = iter_segments(audio_duration)

        print(
            f"\n[{file_index}/{len(audio_files)}] {audio_path.name}: "
            f"{audio_duration:.1f} s -> {len(segments)} candidate 60-second segments"
        )

        file_kept = 0
        file_discarded = 0
        file_skipped = 0
        file_errors = 0

        for segment_index, local_start, local_end in segments:
            segment_duration = local_end - local_start
            absolute_start = absolute_origin + local_start
            absolute_end = absolute_origin + local_end

            events = overlapping_gt(recording_df, absolute_start, absolute_end)
            assessment = evaluate_segment(events, segment_duration)

            site_out = (
                OUT_DIR
                / safe_path_component(site_name)
                / safe_path_component(audio_path.stem)
            )
            token = (
                f"s{segment_index:04d}__"
                f"{absolute_start:010.3f}-{absolute_end:010.3f}"
            ).replace(".", "p")
            out_audio = site_out / f"{audio_path.stem}__{token}{output_extension()}"
            annotation_csv = site_out / f"{audio_path.stem}__{token}_annotations.csv"

            if not assessment["keep"]:
                totals["discarded"] += 1
                file_discarded += 1
                status = "discarded_without_writing_audio"
                decision = "discard"
                out_audio_value = ""
                annotation_csv_value = ""
            elif (
                SKIP_EXISTING
                and not FORCE_REBUILD
                and output_is_valid(out_audio)
            ):
                totals["skipped_existing"] += 1
                file_skipped += 1
                status = "skipped_existing"
                decision = "keep"
                out_audio_value = str(out_audio)
                annotation_csv_value = (
                    str(annotation_csv) if annotation_csv.exists() else ""
                )
            else:
                try:
                    extract_audio_segment(
                        audio_path=audio_path,
                        local_start=local_start,
                        duration=segment_duration,
                        out_audio=out_audio,
                    )
                    if WRITE_CLIP_ANNOTATION_CSV:
                        write_clip_annotations(events, annotation_csv)
                    totals["kept"] += 1
                    file_kept += 1
                    status = "ok"
                    decision = "keep"
                    out_audio_value = str(out_audio)
                    annotation_csv_value = (
                        str(annotation_csv) if WRITE_CLIP_ANNOTATION_CSV else ""
                    )
                except Exception as exc:
                    totals["errors"] += 1
                    file_errors += 1
                    status = f"error: {type(exc).__name__}: {exc}"
                    decision = "error"
                    out_audio_value = ""
                    annotation_csv_value = ""
                    print(
                        f"  [ERROR segment {segment_index:04d} "
                        f"{local_start:.1f}-{local_end:.1f}s] "
                        f"{type(exc).__name__}: {exc}"
                    )

            append_manifest_row(
                {
                    "audio_file": str(audio_path),
                    "recording_id": recording_id,
                    "site": site_name,
                    "match_method": match_method,
                    "origin_method": origin_method,
                    "segment_index": segment_index,
                    "local_start": local_start,
                    "local_end": local_end,
                    "absolute_start": absolute_start,
                    "absolute_end": absolute_end,
                    "event_count": assessment["event_count"],
                    "distinct_label_count": assessment["distinct_label_count"],
                    "labels": assessment["labels"],
                    "labelled_seconds": assessment["labelled_seconds"],
                    "labelled_fraction": assessment["labelled_fraction"],
                    "decision": decision,
                    "decision_reason": assessment["decision_reason"],
                    "status": status,
                    "out_audio": out_audio_value,
                    "annotation_csv": annotation_csv_value,
                }
            )

            if (segment_index + 1) % 20 == 0 or segment_index == len(segments) - 1:
                print(
                    f"  Assessed {segment_index + 1}/{len(segments)} "
                    f"(kept={file_kept}, discarded={file_discarded}, "
                    f"existing={file_skipped}, errors={file_errors})"
                )

    print("\nFinished SERF 60-second GT-filtered extraction.")
    print(f"  Newly written segments: {totals['kept']}")
    print(f"  Discarded by GT:        {totals['discarded']}")
    print(f"  Existing segments:      {totals['skipped_existing']}")
    print(f"  Unmatched audio files:  {totals['unmatched_files']}")
    print(f"  Errors:                 {totals['errors']}")
    print(f"\nManifest: {MANIFEST_CSV.resolve()}")


if __name__ == "__main__":
    main()


Loading SERF annotations: D:\WABAD\SERF\serf_acoustic_study_397-20260715-021743.csv
Loaded 83,563 events from 57 recordings and 6 sites.
Segment length: 60 s | keep rule: any | output: FLAC
Found 6 audio files.
[ERROR] Could not probe 20101016T140000Z_NE_188252.mp3: RuntimeError: ffprobe was not found. Install ffmpeg and ensure ffprobe is on PATH, or set FFPROBE_EXE explicitly.
[ERROR] Could not probe 20121019T200000Z_NE_234234.wav: RuntimeError: ffprobe was not found. Install ffmpeg and ensure ffprobe is on PATH, or set FFPROBE_EXE explicitly.

[3/6] [UNMATCHED] 20101021T140000Z_SW_188329.mp3: skipped because no GT recording could be identified.

[4/6] [UNMATCHED] 20101022T140000Z_SW_188330.mp3: skipped because no GT recording could be identified.
[ERROR] Could not probe 20101023T140000Z_SW_188331.mp3: RuntimeError: ffprobe was not found. Install ffmpeg and ensure ffprobe is on PATH, or set FFPROBE_EXE explicitly.
[ERROR] Could not probe 20101016T140000Z_SE-Frog-Pond_194287.mp3: Runti